# 02 — Data Cleaning (Phase 2)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 2: import และเรียกใช้ฟังก์ชันจาก `src/rri/cleaning/`
เท่านั้น — ไม่มี business logic ในเซลล์ตาม CLAUDE.md N2 ลำดับขั้นตอนคงที่ตามที่ตกลงไว้:

1. Extract + structural fixes (encoding, BOM, header typo, twin-column coalesce, BE→CE)
2. Harmonize category taxonomy (**ก่อน** dedup)
3. Resolve dates ← **CHECKPOINT**
4. Deduplicate
5. Remaining field-level fixes (coordinates, times, injury policy, nulls)
6. Validate + write staging

อ้างอิงข้อเท็จจริงและเหตุผลทั้งหมดจาก Phase 1: [`CLAUDE.md`](../CLAUDE.md) §6 และ
[`docs/data_quality_findings.md`](../docs/data_quality_findings.md)

## 0. Setup

In [1]:
import sys
from datetime import UTC, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.cleaning.extract import (
    add_extract_metadata,
    coalesce_twin_columns,
    convert_buddhist_year,
    pivot_rainfall_leads,
    read_raw_csv,
    rename_headers,
)
from rri.cleaning.log import CleaningLogEntry, cleaning_log_to_frame
from rri.cleaning.taxonomy import apply_taxonomy, load_taxonomy_mapping
from rri.utils.logging import configure_logging
from rri.utils.profiling import find_conflicting_groups

configure_logging()
settings.ensure_writable_dirs()
RUN_TIMESTAMP = datetime.now(UTC)
cleaning_log: list[CleaningLogEntry] = []
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Extract + structural fixes

เริ่มจาก extract ไฟล์ CSV ทั้ง 6 ไฟล์ (ไฟล์ accidents หลักจะผ่านอีก 5 ขั้นตอนถัดไป
ส่วนอีก 5 ไฟล์จบที่ขั้นตอนนี้) ทุกไฟล์อ่านด้วย `read_raw_csv` (encoding ตาม P7.2 —
BOM 5 ใน 6 ไฟล์, `stt_car_fuel` ไม่มี) แล้วเติม `_source_file`/`_ingested_at`/`_row_hash`

In [2]:
accidents_raw = read_raw_csv(settings.raw_dir / "thailand_accidents_2019_2026_combined.csv")
accidents_raw = add_extract_metadata(
    accidents_raw, "thailand_accidents_2019_2026_combined.csv", RUN_TIMESTAMP
)

accidents_step1, twin_logs = coalesce_twin_columns(accidents_raw)

for log_entry in twin_logs:
    cleaning_log.append(
        CleaningLogEntry(
            step="1_extract",
            rule=f"coalesce_twin_columns:{log_entry.old_column}",
            rows_before=log_entry.total_rows,
            rows_affected=log_entry.rows_from_new,
            description=(
                f"{log_entry.old_column} <- {log_entry.new_column}: "
                f"{log_entry.rows_from_old} from old, {log_entry.rows_from_new} from new, "
                f"{log_entry.rows_still_null} still null"
            ),
        )
    )

cleaning_log_to_frame(cleaning_log)

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""1_extract""","""coalesce_twin_columns:รถยนต์นั…",161307,8738,"""รถยนต์นั่งส่วนบุคคล <- รถยนต์น…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกไม่เกิน10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกมากกว่า10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถอื่นๆ""",161307,8738,"""รถอื่นๆ <- อื่นๆ: 133375 from …"


**ยืนยันตรงกับ Phase 1**: overlap ระหว่างคอลัมน์เก่า/ใหม่ทั้ง 4 คู่ = 0 แถว (ถ้าไม่ตรง
`coalesce_twin_columns` จะ raise ทันที ไม่ใช่ coalesce มั่ว ๆ) จำนวนแถวที่มาจากคอลัมน์ใหม่ต่อคู่
ตรงกับสัดส่วน 5.4% ที่ Phase 1 พบ

In [3]:
rainfall_raw = read_raw_csv(settings.raw_dir / "thailand_rainfall_2021_2026_combined.csv")
rainfall_raw = add_extract_metadata(
    rainfall_raw, "thailand_rainfall_2021_2026_combined.csv", RUN_TIMESTAMP
)
rainfall_renamed = rename_headers(rainfall_raw, {"Foreast_Month": "Forecast_Month"})

rainfall_step1 = pivot_rainfall_leads(rainfall_renamed)
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "pivot_rainfall_leads",
        rainfall_renamed.height,
        rainfall_step1.height,
        f"{rainfall_renamed.height} rows (6 leads x group) -> "
        f"{rainfall_step1.height} groups x rain_lead0..5",
    )
)
pl.DataFrame(
    [
        {
            "rows_before_pivot": rainfall_renamed.height,
            "rows_after_pivot": rainfall_step1.height,
            "columns": rainfall_step1.columns,
        }
    ]
)

rows_before_pivot,rows_after_pivot,columns
i64,i64,list[str]
30954,5159,"[""Initial_Year"", ""Initial_Month"", … ""rain_lead5""]"


**ไม่มีแถวหายไป**: `rows_before_pivot / 6 == rows_after_pivot` (6 lead-time ต่อกลุ่มกลายเป็น
1 แถว x 6 คอลัมน์ ไม่ใช่การ dedup) ยืนยันตาม P7.5

In [4]:
registration_raw = read_raw_csv(settings.raw_dir / "car_registered_new_2565_2569_combined.csv")
registration_raw = add_extract_metadata(
    registration_raw, "car_registered_new_2565_2569_combined.csv", RUN_TIMESTAMP
)
registration_step1 = convert_buddhist_year(registration_raw, "ปี พ.ศ.")
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "convert_buddhist_year",
        registration_step1.height,
        registration_step1.height,
        "added year_ce = ปี พ.ศ. - 543 for every row (original column kept)",
    )
)
registration_step1.select("ปี พ.ศ.", "year_ce").unique().sort("ปี พ.ศ.")

ปี พ.ศ.,year_ce
str,i64
"""2565""",2022
"""2566""",2023
"""2567""",2024
"""2568""",2025
"""2569""",2026


**ตรวจแล้ว**: พ.ศ. 2565–2569 กลายเป็น ค.ศ. 2022–2026 ตรงตามสูตร `CE = BE - 543`
คอลัมน์เดิมยังอยู่ครบ (ไม่ทำลายข้อมูลดิบ)

In [5]:
flood_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_flood_risk_areas.csv"),
    "thailand_flood_risk_areas.csv",
    RUN_TIMESTAMP,
)
heavy_rain_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_heavy_rain_R35mm.csv"),
    "thailand_heavy_rain_R35mm.csv",
    RUN_TIMESTAMP,
)
stt_car_fuel_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "stt_car_fuel_at_25690228.csv"),
    "stt_car_fuel_at_25690228.csv",
    RUN_TIMESTAMP,
)

pl.DataFrame(
    [
        {"file": "flood_risk", "rows": flood_raw.height, "cols": flood_raw.width},
        {"file": "heavy_rain_r35mm", "rows": heavy_rain_raw.height, "cols": heavy_rain_raw.width},
        {"file": "stt_car_fuel", "rows": stt_car_fuel_raw.height, "cols": stt_car_fuel_raw.width},
    ]
)

file,rows,cols
str,i64,i64
"""flood_risk""",27023,16
"""heavy_rain_r35mm""",3465,7
"""stt_car_fuel""",4643,8


ไฟล์อีก 3 ไฟล์นี้ไม่มี structural fix เฉพาะใน Step 1 (นอกจาก encoding/metadata) —
เขียนลง staging ตรง ๆ

In [6]:
accidents_step1.write_parquet(settings.staging_dir / "accidents.parquet")
rainfall_step1.write_parquet(settings.staging_dir / "rainfall.parquet")
registration_step1.write_parquet(settings.staging_dir / "car_registered_new.parquet")
flood_raw.write_parquet(settings.staging_dir / "flood_risk.parquet")
heavy_rain_raw.write_parquet(settings.staging_dir / "heavy_rain_r35mm.parquet")
stt_car_fuel_raw.write_parquet(settings.staging_dir / "stt_car_fuel.parquet")

sorted(p.name for p in settings.staging_dir.glob("*.parquet"))

['accidents.parquet',
 'car_registered_new.parquet',
 'flood_risk.parquet',
 'heavy_rain_r35mm.parquet',
 'rainfall.parquet',
 'stt_car_fuel.parquet']

**Step 1 เสร็จสมบูรณ์** — เขียนไฟล์ทั้ง 6 ลง `data/10_staging/` แล้ว `accidents.parquet`
จะถูกเขียนทับอีกครั้งใน Step 6 หลังผ่าน Step 2–5 ครบ ไฟล์ที่เหลือจบที่นี่ (ไม่มีขั้นตอนถัดไปแตะต้อง)

## 2. Harmonize category taxonomy (ก่อน dedup)

โหลด `accidents.parquet` จาก Step 1 แล้ว apply mapping table ทั้ง 3 ตัว
(`config/mappings/{cause,crash_type,road_geometry}_groups.csv` — สร้างโดย
`scripts/build_taxonomy_mappings.py`, ตรวจสอบด้วยมือแล้ว) **ต้องทำก่อน Step 4 (dedup)**
เพราะ P5 พบว่า 74% ของ "conflict" ในคู่ duplicate ที่คอลัมน์ `บริเวณที่เกิดเหตุ` จริง ๆ
เป็นแค่คำพ้องความหมายคนละชุด ไม่ใช่ conflict จริง

In [7]:
accidents_step1 = pl.read_parquet(settings.staging_dir / "accidents.parquet")

cause_mapping = load_taxonomy_mapping(settings.mappings_dir / "cause_groups.csv")
crash_type_mapping = load_taxonomy_mapping(settings.mappings_dir / "crash_type_groups.csv")
road_geometry_mapping = load_taxonomy_mapping(settings.mappings_dir / "road_geometry_groups.csv")

accidents_step2, cause_log = apply_taxonomy(accidents_step1, "มูลเหตุสันนิษฐาน", cause_mapping)
accidents_step2, crash_type_log = apply_taxonomy(
    accidents_step2, "ลักษณะการเกิดเหตุ", crash_type_mapping
)
accidents_step2, geometry_log = apply_taxonomy(
    accidents_step2, "บริเวณที่เกิดเหตุ", road_geometry_mapping
)
accidents_step2, geometry_secondary_log = apply_taxonomy(
    accidents_step2,
    "บริเวณที่เกิดเหตุอื่น",
    road_geometry_mapping,
    new_column="บริเวณที่เกิดเหตุอื่น_group",
)

for taxonomy_log in (cause_log, crash_type_log, geometry_log, geometry_secondary_log):
    cleaning_log.append(
        CleaningLogEntry(
            "2_taxonomy",
            f"apply_taxonomy:{taxonomy_log.column}",
            taxonomy_log.total_rows,
            taxonomy_log.rows_mapped,
            f"mapped={taxonomy_log.rows_mapped} null={taxonomy_log.rows_null} "
            f"unmapped={taxonomy_log.rows_unmapped} {taxonomy_log.unmapped_values[:5]}",
        )
    )

pl.DataFrame(
    [
        {
            "column": log_entry.column,
            "rows_mapped": log_entry.rows_mapped,
            "rows_null": log_entry.rows_null,
            "rows_unmapped": log_entry.rows_unmapped,
            "unmapped_sample": log_entry.unmapped_values[:5],
        }
        for log_entry in (cause_log, crash_type_log, geometry_log, geometry_secondary_log)
    ]
)

column,rows_mapped,rows_null,rows_unmapped,unmapped_sample
str,i64,i64,i64,list[null]
"""มูลเหตุสันนิษฐาน""",158627,2680,0,[]
"""ลักษณะการเกิดเหตุ""",153244,8063,0,[]
"""บริเวณที่เกิดเหตุ""",147866,13441,0,[]
"""บริเวณที่เกิดเหตุอื่น""",8462,152845,0,[]


**ตรวจ**: `rows_unmapped` ต้องเป็น 0 ทุกคอลัมน์ (mapping table ครอบคลุมทุกค่าที่พบจริงใน
ข้อมูล ณ ตอนสร้างตาราง) ถ้าไม่เป็น 0 แปลว่าข้อมูลจริงมีค่าที่ mapping table ไม่รู้จัก — ต้อง
หยุดแล้วรายงาน ไม่ใช่เดาแล้วปล่อยผ่าน

In [8]:
geometry_conflicts_before = find_conflicting_groups(accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ")
geometry_conflicts_after = find_conflicting_groups(
    accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ_group"
)

dup_groups = (
    accidents_step2.filter(pl.col("ACC_CODE").is_not_null())
    .group_by("ACC_CODE")
    .len()
    .filter(pl.col("len") > 1)
)

pl.DataFrame(
    [
        {
            "duplicate_groups_checked": dup_groups.height,
            "conflicts_before_harmonization": len(geometry_conflicts_before),
            "conflict_rate_before_pct": round(
                len(geometry_conflicts_before) / dup_groups.height * 100, 1
            ),
            "conflicts_after_harmonization": len(geometry_conflicts_after),
            "conflict_rate_after_pct": round(
                len(geometry_conflicts_after) / dup_groups.height * 100, 1
            ),
        }
    ]
)

duplicate_groups_checked,conflicts_before_harmonization,conflict_rate_before_pct,conflicts_after_harmonization,conflict_rate_after_pct
i64,i64,f64,i64,f64
1641,1216,74.1,52,3.2


**เทียบกับ Phase 1**: conflict rate เดิม (raw `บริเวณที่เกิดเหตุ`) ควรตรงกับ P5 ที่ 74%
(1,216/1,641) หลัง harmonize ด้วย `บริเวณที่เกิดเหตุ_group` (group ตามประเภทถนน ตัด slope
descriptor ทิ้ง) ควรลดลงเหลือแค่ residual ที่เป็น conflict จริง (ประเภทถนนต่างกันจริง ไม่ใช่แค่
คำอธิบายความชันต่างกัน)

In [9]:
accidents_step2.write_parquet(settings.staging_dir / "accidents.parquet")
cleaning_log_to_frame(cleaning_log).filter(pl.col("step") == "2_taxonomy")

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""2_taxonomy""","""apply_taxonomy:มูลเหตุสันนิษฐา…",161307,158627,"""mapped=158627 null=2680 unmapp…"
"""2_taxonomy""","""apply_taxonomy:ลักษณะการเกิดเห…",161307,153244,"""mapped=153244 null=8063 unmapp…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,147866,"""mapped=147866 null=13441 unmap…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,8462,"""mapped=8462 null=152845 unmapp…"


**Step 2 เสร็จสมบูรณ์** — เขียนทับ `accidents.parquet` ด้วยเวอร์ชันที่มี taxonomy
harmonized แล้ว (`_group` columns เพิ่มเข้ามา คอลัมน์ raw เดิมยังอยู่ครบ) พร้อมสำหรับ Step 3

## 3. Resolve dates — CHECKPOINT

## 4. Deduplicate

## 5. Remaining field-level fixes

## 6. Validate + write staging